In [3]:
import pandas as pd
import os
import time
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from tqdm import tqdm
from IPython.display import clear_output

# --- KONFIGURASI ---
BASE_NOISE_DIR = r'D:\waveform_data_noise'
CATALOG_PATH = r'D:\seismiccode\usgs_katalog\katalog_usgs_master_2001_2025.csv'
SYSTEM_LOG = os.path.join(BASE_NOISE_DIR, 'system_log_modern_noise.log')
FAILED_LOG = os.path.join(BASE_NOISE_DIR, 'failed_log_modern_noise.csv')

# Hanya stasiun Indonesia (BMKG + GEOFON)
STRICT_NETWORKS = ["IA", "GE"]

def run_resilient_noise_downloader_with_progress():
    client = Client("EARTHSCOPE", timeout=180)

    # Load katalog
    df = pd.read_csv(CATALOG_PATH)
    df['time'] = pd.to_datetime(df['time'], format='ISO8601')

    # Filter wilayah Indonesia
    mask_indo = (df['latitude'].between(-15, 10)) & (df['longitude'].between(90, 150))
    df_indo = df[(df['time'].dt.year >= 2015) & (df['mag'] >= 4.5) & mask_indo].copy()
    df_indo = df_indo.sort_values(by='time', ascending=False)

    total_target = len(df_indo)

    pbar = tqdm(total=total_target, desc="🌑 Downloading Noise (3-Comp)")

    success_count = 0
    skip_count = 0
    failed_rows = []

    for idx, row in df_indo.iterrows():
        year_dir = os.path.join(BASE_NOISE_DIR, str(row['time'].year))
        os.makedirs(year_dir, exist_ok=True)

        noise_id = f"NOISE_3C_{row['time'].strftime('%Y%m%d_%H%M%S')}"
        file_path = os.path.join(year_dir, f"{noise_id}.mseed")
        empty_marker = file_path + ".empty"

        # Auto-resume
        if os.path.exists(file_path) or os.path.exists(empty_marker):
            pbar.update(1)
            skip_count += 1
            continue

        # 1 jam sebelum origin
        noise_start = UTCDateTime(row['time']) - 3600
        noise_end = noise_start + 30

        retry_count = 0
        while retry_count < 3:
            try:
                bulk = [(net, "*", "*", "BH?", noise_start, noise_end) for net in STRICT_NETWORKS]
                st = client.get_waveforms_bulk(bulk)

                if len(st) > 0:
                    st.merge(method=1, fill_value='interpolate')
                    st.detrend("linear")
                    st.write(file_path, format="MSEED")
                    success_count += 1
                else:
                    with open(empty_marker, 'w') as f:
                        f.write("no data")

                pbar.update(1)
                break

            except Exception as e:
                retry_count += 1
                if retry_count == 3:
                    failed_rows.append({
                        "time": row['time'],
                        "mag": row['mag'],
                        "error": str(e)
                    })
                if "429" in str(e):
                    time.sleep(60)
                else:
                    time.sleep(5)

        pbar.set_postfix({"Success": success_count, "Skipped": skip_count})
        time.sleep(1.2)

    pbar.close()

    if failed_rows:
        pd.DataFrame(failed_rows).to_csv(FAILED_LOG, index=False)

    print(f"\n✅ Selesai! Total Sukses: {success_count}, Total Dilewati: {skip_count}")

if __name__ == "__main__":
    run_resilient_noise_downloader_with_progress()




c:\Users\Very\miniconda3\envs\waveform\lib\site-packages\obspy\io\mseed\core.py:1034: UserWarning: The encoding specified in trace.stats.mseed.encoding does not match the dtype of the data.
A suitable encoding will be chosen.
  warnings.warn(msg, UserWarning)





































































































































































































































































































































































































































































































































































































































































































































































KeyboardInterrupt: 